# HW2 · Super-Convergence на Kaggle
[Статья](https://arxiv.org/abs/1708.07120) · [авторский код](https://github.com/lnsmith54/super-convergence)

Самостоятельный notebook для нашей адаптации эксперимента: один LR-пилот и три режима для seeds **42, 43, 44**. Исходники встроены. Добавьте **CIFAR-10 Python** (`pankrzysiu/cifar10-python`) через Add Input, включите GPU и выберите **Save Version → Save & Run All**.

LR выбирается один раз по validation пилота seed 42 и фиксируется для всех повторов. Test не участвует в подборе. Уже завершённые прогоны с теми же настройками пропускаются; незавершённые продолжаются с checkpoint. На исходных запусках max LR составил **1.0**; новые результаты следует сохранять отдельно от представленных в отчёте.


In [ ]:
from pathlib import Path
import json, os, sys, subprocess, shutil, hashlib
import torch

WORKDIR = Path('/kaggle/working/hw2')
CIFAR_INPUTS = sorted(Path('/kaggle/input').rglob('cifar-10-batches-py'))
if not CIFAR_INPUTS:
    raise RuntimeError('Attach the CIFAR-10 Python dataset through Kaggle Input first.')
DATA_ROOT = CIFAR_INPUTS[0].parent
print('CIFAR-10 source:', DATA_ROOT)
SEEDS = [42, 43, 44]
EXPERIMENTS = ['baseline_long', 'baseline_short', 'onecycle']

STOP_AFTER_EPOCH = None

if not torch.cuda.is_available():
    raise RuntimeError('Включите GPU в настройках Kaggle перед обучением.')
print('Python:', sys.version.split()[0], 'PyTorch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0), 'CUDA:', torch.version.cuda)

WORKDIR.mkdir(parents=True, exist_ok=True)
(WORKDIR / "src").mkdir(exist_ok=True)
SOURCE_SHA256 = {'src/__init__.py': '01ba4719c80b6fe911b091a7c05124b64eeece964e09c058ef8f9805daca546b',
 'src/models.py': 'c1720e89e0345a84f0b38c4d14df6f5721ca99f480ef93ba5a0cb60b9f7a6f9e',
 'src/data.py': '86a8d768fd15b9354b6c6028eb98911ae03bf680e7fdf3b52b3ca8f4070886f8',
 'src/schedules.py': 'f764cb0dfcfc617acc5e99ac36ee2c40ad4d581b4dc431bd3945c03b52c7ac7a',
 'src/train.py': '2ce3e6be48e43a577b9915de8057990e5d175b701a4559de87837d08e9633532',
 'src/analyze.py': '3b55201931c68791e90a98a2f300b3e4c7859dd9befbfaeac66a87c418658eb4'}
for name, expected_hash in SOURCE_SHA256.items():
    path = WORKDIR / name
    if path.exists() and hashlib.sha256(path.read_bytes()).hexdigest() != expected_hash:
        raise RuntimeError(f'Отличается версия {name}. Используйте новую WORKDIR.')
(WORKDIR / 'src/__init__.py').write_text('\n')
os.chdir(WORKDIR)


## Статья и выполненная адаптация
Объяснение метода основано на [статье, версия v3](https://arxiv.org/abs/1708.07120v3). Этот notebook сохраняет настройки уже выполненных запусков. Он не является точным повторением основного примера авторов.

| Компонент | Статья: CIFAR-10 / ResNet-56 | Наш запуск |
|---|---|---|
| Реализация | Caffe, приложение A.4 | PyTorch, Tesla T4 |
| Данные для обучения | 50 000, таблица 6 | 45 000 train / 5 000 validation |
| Бюджеты | около 80 000 / 10 000 шагов, рисунок 1a | 10 560 / 2 112 шагов (120 / 24 эпохи) |
| Batch | 1000 для основного примера; 512 также исследован в таблице 7 | 512 |
| Большой LR | 0.1–3.0, таблица 6 | 0.1–1.0, выбран по нашему пилоту |
| Momentum | 0.9 для основного примера, таблица 8 | 0.95–0.85–0.95 для 1cycle |
| Уменьшение LR | один цикл и завершение с малым LR, раздел 3 | фазы 45%/45%/10%, конечный LR 0.0001 |
| Сравнение скорости | шаги и test accuracy, рисунок 1a | дополнительно время до 85% validation |

Seeds 42/43/44, выделенный validation, правило выбора LR и порог 85% заданы в нашей реализации; статья не задаёт их для этого сравнения. Результаты notebook являются нашими измерениями и не подменяют опубликованные результаты.


## Настройки нашего запуска
Это настройки выполненной адаптации для Kaggle, а не конфигурация основного эксперимента статьи. При продолжении запуска ранее выбранные значения сохраняются.


In [ ]:
DEFAULT_CONFIG = {'protocol': 'kaggle-adaptation-v1',
 'depth': 56,
 'batch_size': 512,
 'bn_momentum': 0.1,
 'num_workers': 2,
 'seeds': [42, 43, 44],
 'target_val_accuracy': 0.85,
 'experiments': {'baseline_long': {'schedule': 'multistep',
                                   'epochs': 120,
                                   'lr': 0.1,
                                   'momentum': 0.9,
                                   'weight_decay': 0.0001,
                                   'milestones': [0.5, 0.75],
                                   'gamma': 0.1},
                 'baseline_short': {'schedule': 'multistep',
                                    'epochs': 24,
                                    'lr': 0.1,
                                    'momentum': 0.9,
                                    'weight_decay': 0.0001,
                                    'milestones': [0.5, 0.75],
                                    'gamma': 0.1},
                 'onecycle': {'schedule': 'onecycle',
                              'epochs': 24,
                              'lr_start': 0.1,
                              'lr_max': 1.0,
                              'lr_final': 0.0001,
                              'up_fraction': 0.45,
                              'down_end': 0.9,
                              'momentum_max': 0.95,
                              'momentum_min': 0.85,
                              'weight_decay': 0.0001},
                 'onecycle_low_lr': {'schedule': 'onecycle',
                                     'epochs': 24,
                                     'lr_start': 0.01,
                                     'lr_max': 0.1,
                                     'lr_final': 1e-05,
                                     'up_fraction': 0.45,
                                     'down_end': 0.9,
                                     'momentum_max': 0.95,
                                     'momentum_min': 0.85,
                                     'weight_decay': 0.0001},
                 'onecycle_high_wd': {'schedule': 'onecycle',
                                      'epochs': 24,
                                      'lr_start': 0.1,
                                      'lr_max': 1.0,
                                      'lr_final': 0.0001,
                                      'up_fraction': 0.45,
                                      'down_end': 0.9,
                                      'momentum_max': 0.95,
                                      'momentum_min': 0.85,
                                      'weight_decay': 0.001}},
 'lr_range': {'schedule': 'range',
              'steps': 1000,
              'lr_start': 0.01,
              'lr_end': 2.0,
              'weight_decay': 0.0001,
              'validate_every': 50}}
config_path = WORKDIR / 'configs/kaggle.json'
config_path.parent.mkdir(exist_ok=True)
if not config_path.exists():
    config_path.write_text(json.dumps(DEFAULT_CONFIG, indent=2))


## Модель ResNet-56


In [ ]:
%%writefile {WORKDIR}/src/models.py
import torch
from torch import nn


class Block(nn.Module):
    def __init__(self, cin, cout, stride=1, bn_momentum=0.1):
        super().__init__()
        self.conv1 = nn.Conv2d(cin, cout, 3, stride, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(cout, momentum=bn_momentum)
        self.conv2 = nn.Conv2d(cout, cout, 3, 1, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(cout, momentum=bn_momentum)
        self.shortcut = nn.Identity() if cin == cout and stride == 1 else nn.Sequential(
            nn.Conv2d(cin, cout, 1, stride, bias=False),
            nn.BatchNorm2d(cout, momentum=bn_momentum),
        )

    def forward(self, x):
        z = torch.relu(self.bn1(self.conv1(x)))
        z = self.bn2(self.conv2(z))
        return torch.relu(z + self.shortcut(x))


class CifarResNet(nn.Module):
    def __init__(self, depth=56, bn_momentum=0.1):
        super().__init__()
        if depth < 8 or (depth - 2) % 6:
            raise ValueError('depth must have the form 6n+2')
        n = (depth - 2) // 6
        self.stem = nn.Sequential(nn.Conv2d(3, 16, 3, 1, 1, bias=False),
                                  nn.BatchNorm2d(16, momentum=bn_momentum), nn.ReLU())
        blocks, cin = [], 16
        for stage, cout in enumerate((16, 32, 64)):
            for i in range(n):
                blocks.append(Block(cin, cout, 2 if stage and i == 0 else 1, bn_momentum))
                cin = cout
        self.blocks = nn.Sequential(*blocks)
        self.head = nn.Linear(64, 10)
        for module in self.modules():
            if isinstance(module, nn.Conv2d):
                nn.init.kaiming_normal_(module.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(module, nn.BatchNorm2d):
                nn.init.ones_(module.weight)
                nn.init.zeros_(module.bias)

    def forward(self, x):
        return self.head(self.blocks(self.stem(x)).mean((2, 3)))


## Данные CIFAR-10


In [ ]:
%%writefile {WORKDIR}/src/data.py
import random
import numpy as np
import torch
from torch.utils.data import DataLoader, Subset, TensorDataset


def seed_worker(_):
    seed = torch.initial_seed() % 2**32
    random.seed(seed)
    np.random.seed(seed)


def datasets(root, synthetic=False):
    if synthetic:
        g = torch.Generator().manual_seed(2026)
        ds = TensorDataset(torch.randn(96, 3, 32, 32, generator=g),
                           torch.randint(0, 10, (96,), generator=g))
        return Subset(ds, range(64)), Subset(ds, range(64, 80)), Subset(ds, range(80, 96))
    from torchvision import datasets as tvd, transforms as T


    tvd.CIFAR10.url = 'https://cave.cs.toronto.edu/kriz/cifar-10-python.tar.gz'

    norm = T.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616))
    train_tf = T.Compose([T.RandomCrop(32, padding=4), T.RandomHorizontalFlip(), T.ToTensor(), norm])
    eval_tf = T.Compose([T.ToTensor(), norm])
    augmented = tvd.CIFAR10(root, train=True, download=True, transform=train_tf)
    plain = tvd.CIFAR10(root, train=True, download=False, transform=eval_tf)
    labels = np.asarray(plain.targets)
    rng = np.random.default_rng(2026)
    train_idx, val_idx = [], []
    for label in range(10):
        idx = rng.permutation(np.flatnonzero(labels == label))
        val_idx.extend(idx[:500].tolist())
        train_idx.extend(idx[500:].tolist())
    test = tvd.CIFAR10(root, train=False, download=True, transform=eval_tf)
    return Subset(augmented, train_idx), Subset(plain, val_idx), test


def loader(ds, batch_size, workers, seed, shuffle, cuda):
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, drop_last=False,
                      num_workers=workers, pin_memory=cuda, worker_init_fn=seed_worker,
                      generator=torch.Generator().manual_seed(seed))


## Изменение learning rate и momentum в ходе обучения


In [ ]:
%%writefile {WORKDIR}/src/schedules.py
def interpolate(a, b, fraction):
    return a + (b - a) * fraction


def schedule(config, step, total_steps):
    if total_steps < 2 or not 0 <= step < total_steps:
        raise ValueError('need at least two updates and a valid zero-based step')
    p = step / (total_steps - 1)
    kind = config['schedule']
    if kind == 'range':
        return interpolate(config['lr_start'], config['lr_end'], p), 0.9
    if kind == 'multistep':
        decays = sum(p >= x for x in config.get('milestones', [0.5, 0.75]))
        return config['lr'] * config.get('gamma', 0.1) ** decays, config.get('momentum', 0.9)
    if kind != 'onecycle':
        raise ValueError(f'unknown schedule: {kind}')
    up, down = config.get('up_fraction', 0.45), config.get('down_end', 0.9)
    lo, hi, final = config['lr_start'], config['lr_max'], config['lr_final']
    mh, ml = config.get('momentum_max', 0.95), config.get('momentum_min', 0.85)
    if not 0 < up < down < 1:
        raise ValueError('onecycle phase boundaries must satisfy 0 < up < down < 1')
    if p <= up:
        return interpolate(lo, hi, p / up), interpolate(mh, ml, p / up)
    if p <= down:
        q = (p - up) / (down - up)
        return interpolate(hi, lo, q), interpolate(ml, mh, q)
    return interpolate(lo, final, (p - down) / (1 - down)), mh


## Обучение и оценка


In [ ]:
%%writefile {WORKDIR}/src/train.py
import argparse
import csv
import hashlib
from importlib.metadata import version, PackageNotFoundError
import json
import math
import os
from pathlib import Path
import platform
import random
import time

import numpy as np
import torch
from torch import nn
from .models import CifarResNet
from .data import datasets, loader
from .schedules import schedule


def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False


def sync(device):
    if device.type == 'cuda':
        torch.cuda.synchronize(device)


def atomic_json(path, obj):
    tmp = path.with_suffix('.tmp')
    tmp.write_text(json.dumps(obj, indent=2, ensure_ascii=False), encoding='utf-8')
    os.replace(tmp, path)


def write_csv(path, rows):
    if not rows:
        return
    tmp = path.with_suffix('.tmp')
    with tmp.open('w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)
    os.replace(tmp, path)


@torch.inference_mode()
def evaluate(model, batches, device):
    model.eval()
    loss_sum, correct, count = 0.0, 0, 0
    for x, y in batches:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        logits = model(x)
        loss_sum += nn.functional.cross_entropy(logits, y, reduction='sum').item()
        correct += (logits.argmax(1) == y).sum().item()
        count += len(y)
    return loss_sum / count, correct / count


def metadata(config, experiment, seed, device, synthetic):
    source_hash = hashlib.sha256()
    for path in sorted(Path(__file__).parent.glob('*.py')):
        source_hash.update(path.name.encode())
        source_hash.update(path.read_bytes())
    try:
        torchvision_version = version('torchvision')
    except PackageNotFoundError:
        torchvision_version = None
    return dict(config=config, experiment=experiment, seed=seed, synthetic=synthetic,
                source_sha256=source_hash.hexdigest(), python=platform.python_version(),
                torch=torch.__version__, torchvision=torchvision_version, numpy=np.__version__, cuda=torch.version.cuda,
                cudnn=torch.backends.cudnn.version(), device=str(device),
                gpu=torch.cuda.get_device_name(device) if device.type == 'cuda' else None,
                precision='FP32', split_seed=2026, test_selection='best validation accuracy',
                train_size=64 if synthetic else 45000, val_size=16 if synthetic else 5000,
                test_size=16 if synthetic else 10000)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--config', default='configs/kaggle.json')
    ap.add_argument('--experiment', default='onecycle')
    ap.add_argument('--seed', type=int, default=42)
    ap.add_argument('--data-root', default='data')
    ap.add_argument('--output', default='results')
    ap.add_argument('--device', default='auto')
    ap.add_argument('--resume', action='store_true')
    ap.add_argument('--stop-after-epoch', type=int, help='save a partial run for later resume')
    ap.add_argument('--synthetic', action='store_true', help='software smoke test only')
    args = ap.parse_args()
    all_cfg = json.loads(Path(args.config).read_text())
    exp = all_cfg['lr_range'] if args.experiment == 'lr_range' else all_cfg['experiments'][args.experiment]
    config = {k: v for k, v in all_cfg.items() if k not in ('experiments', 'lr_range')}
    config.update(exp)
    if args.synthetic:
        config.update(depth=8, batch_size=16, num_workers=0)
        if args.experiment == 'lr_range':
            config.update(steps=8, validate_every=2)
        else:
            config['epochs'] = 2
    device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu') if args.device == 'auto' else torch.device(args.device)
    seed_all(args.seed)
    out = Path(args.output) / f'{args.experiment}_seed{args.seed}'

    if args.synthetic and 'results' in Path(args.output).parts:
        raise ValueError('choose a separate --output directory for synthetic smoke tests')
    out.mkdir(parents=True, exist_ok=True)
    if (out / 'summary.json').exists():
        raise FileExistsError(f'{out} already completed. Choose a new output path.')
    if any(out.iterdir()) and not args.resume:
        raise FileExistsError(f'{out} exists. Use --resume or a new output path.')
    train_ds, val_ds, test_ds = datasets(args.data_root, args.synthetic)
    batches_per_epoch = math.ceil(len(train_ds) / config['batch_size'])
    total_steps = config['steps'] if args.experiment == 'lr_range' else config['epochs'] * batches_per_epoch
    val_loader = loader(val_ds, config['batch_size'], config['num_workers'], 2026, False, device.type == 'cuda')
    model = CifarResNet(config['depth'], config['bn_momentum']).to(device)
    optimizer = torch.optim.SGD(model.parameters(), lr=schedule(config, 0, total_steps)[0],
                                momentum=0.9, weight_decay=config['weight_decay'])
    meta = metadata(config, args.experiment, args.seed, device, args.synthetic)
    meta['parameter_count'] = sum(x.numel() for x in model.parameters())
    meta['steps_per_epoch'] = batches_per_epoch
    if args.experiment == 'lr_range':
        if args.resume:
            raise ValueError('LR range test has no resume. Choose a fresh output path.')
        atomic_json(out / 'metadata.json', meta)
        records, step, smooth, best, diverged = [], 0, 0.0, float('inf'), False
        sync(device)
        range_started = time.perf_counter()
        while step < total_steps and not diverged:
            train_loader = loader(train_ds, config['batch_size'], config['num_workers'],
                                  args.seed + step // batches_per_epoch, True, device.type == 'cuda')
            for x, y in train_loader:
                lr, momentum = schedule(config, step, total_steps)
                for group in optimizer.param_groups:
                    group.update(lr=lr, momentum=momentum)
                model.train()
                optimizer.zero_grad(set_to_none=True)
                loss = nn.functional.cross_entropy(model(x.to(device)), y.to(device))
                raw = loss.item()
                if not math.isfinite(raw):
                    diverged = True
                    break
                loss.backward()
                optimizer.step()
                smooth = 0.98 * smooth + 0.02 * raw
                corrected = smooth / (1 - 0.98 ** (step + 1))
                best = min(best, corrected)
                row = dict(step=step + 1, lr=lr, loss=raw, smooth_loss=corrected, val_loss='', val_accuracy='')
                if (step + 1) % config['validate_every'] == 0 or step + 1 == total_steps:
                    row['val_loss'], row['val_accuracy'] = evaluate(model, val_loader, device)
                    print(f'LR range {step+1}/{total_steps}, lr={lr:.4g}, val={row["val_accuracy"]:.3f}', flush=True)
                records.append(row)
                step += 1
                if step >= 20 and corrected > 4 * best:
                    diverged = True
                if step >= total_steps or diverged:
                    break
        write_csv(out / 'lr_range.csv', records)
        sync(device)
        atomic_json(out / 'summary.json', dict(experiment=args.experiment, seed=args.seed,
                    synthetic=args.synthetic, updates=step, diverged=diverged, test_evaluated=False,
                    elapsed_seconds=time.perf_counter() - range_started))
        return
    rows, best_accuracy, best_epoch = [], -1.0, 0
    train_seconds, total_seconds, start_epoch = 0.0, 0.0, 0
    checkpoint_path = out / 'last.pt'
    if args.resume:
        checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
        old_meta = json.loads((out / 'metadata.json').read_text())
        for field in ('config', 'seed', 'source_sha256', 'synthetic', 'experiment', 'torch', 'torchvision', 'gpu'):
            if old_meta[field] != meta[field]:
                raise ValueError(f'resume mismatch: {field}')
        model.load_state_dict(checkpoint['model'])
        optimizer.load_state_dict(checkpoint['optimizer'])
        rows, start_epoch = checkpoint['rows'], checkpoint['epoch']
        best_accuracy, best_epoch = checkpoint['best_accuracy'], checkpoint['best_epoch']
        train_seconds, total_seconds = checkpoint['train_seconds'], checkpoint['total_seconds']
        torch.set_rng_state(checkpoint['torch_rng'])
        if device.type == 'cuda':
            torch.cuda.set_rng_state_all(checkpoint['cuda_rng'])
        random.setstate(checkpoint['random_rng'])
        np.random.set_state(checkpoint['numpy_rng'])

        write_csv(out / 'history.csv', rows)
    else:
        atomic_json(out / 'metadata.json', meta)
    for epoch in range(start_epoch, config['epochs']):

        seed_all(args.seed + epoch)
        batches = loader(train_ds, config['batch_size'], config['num_workers'], args.seed + epoch,
                         True, device.type == 'cuda')
        model.train()
        sync(device)
        epoch_started = time.perf_counter()
        loss_sum, correct, count = 0.0, 0, 0
        for batch, (x, y) in enumerate(batches):
            step = epoch * batches_per_epoch + batch
            lr, momentum = schedule(config, step, total_steps)
            for group in optimizer.param_groups:
                group.update(lr=lr, momentum=momentum)
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            logits = model(x)
            loss = nn.functional.cross_entropy(logits, y)
            if not math.isfinite(loss.item()):
                atomic_json(out / 'failure.json', dict(epoch=epoch + 1, step=step + 1, lr=lr, reason='nonfinite loss'))
                raise RuntimeError(f'divergence at update {step+1}, lr={lr}')
            loss.backward()
            optimizer.step()
            loss_sum += loss.item() * len(y)
            correct += (logits.detach().argmax(1) == y).sum().item()
            count += len(y)
        sync(device)
        train_seconds += time.perf_counter() - epoch_started
        val_loss, val_accuracy = evaluate(model, val_loader, device)
        sync(device)
        total_seconds += time.perf_counter() - epoch_started
        row = dict(epoch=epoch + 1, step=(epoch + 1) * batches_per_epoch,
                   lr=lr, momentum=momentum, train_loss=loss_sum / count,
                   train_accuracy=correct / count, val_loss=val_loss, val_accuracy=val_accuracy,
                   train_seconds=train_seconds, elapsed_seconds=total_seconds)
        rows.append(row)

        if val_accuracy > best_accuracy:
            best_accuracy, best_epoch = val_accuracy, epoch + 1
            best_model = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        elif epoch == start_epoch and args.resume:
            best_model = checkpoint['best_model']
        checkpoint = dict(epoch=epoch + 1, model=model.state_dict(), optimizer=optimizer.state_dict(),
                          best_model=best_model, best_accuracy=best_accuracy, best_epoch=best_epoch,
                          rows=rows, train_seconds=train_seconds, total_seconds=total_seconds,
                          torch_rng=torch.get_rng_state(), cuda_rng=torch.cuda.get_rng_state_all() if device.type == 'cuda' else [],
                          random_rng=random.getstate(), numpy_rng=np.random.get_state())
        tmp = out / 'last.tmp'
        torch.save(checkpoint, tmp)
        os.replace(tmp, checkpoint_path)
        write_csv(out / 'history.csv', rows)
        print(f'{args.experiment} seed={args.seed} epoch={epoch+1}/{config["epochs"]} '
              f'val={val_accuracy:.4f} time={total_seconds:.1f}s', flush=True)
        if args.stop_after_epoch and epoch + 1 >= args.stop_after_epoch and epoch + 1 < config['epochs']:
            print('Partial run saved. Continue with --resume.', flush=True)
            return

    if start_epoch == config['epochs']:
        best_model = checkpoint['best_model']
    final_val_accuracy = rows[-1]['val_accuracy']
    model.load_state_dict(best_model)
    test_loader = loader(test_ds, config['batch_size'], config['num_workers'], 2026, False, device.type == 'cuda')
    test_loss, test_accuracy = evaluate(model, test_loader, device)
    reached = next((r for r in rows if r['val_accuracy'] >= config['target_val_accuracy']), None)
    atomic_json(out / 'summary.json', dict(experiment=args.experiment, seed=args.seed, synthetic=args.synthetic,
                epochs=config['epochs'], updates=total_steps, final_val_accuracy=final_val_accuracy,
                best_val_accuracy=best_accuracy, selected_epoch=best_epoch,
                test_accuracy=test_accuracy, test_loss=test_loss,
                train_seconds=train_seconds, elapsed_seconds=total_seconds,
                target_val_accuracy=config['target_val_accuracy'],
                steps_to_target=reached['step'] if reached else None,
                seconds_to_target=reached['elapsed_seconds'] if reached else None))


if __name__ == '__main__':
    main()


## Анализ результатов


In [ ]:
%%writefile {WORKDIR}/src/analyze.py
import argparse
import csv
import json
from pathlib import Path
import statistics
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--results', default='results')
    args = ap.parse_args()
    root = Path(args.results)
    runs = []
    for path in sorted(root.glob('*/summary.json')):
        summary = json.loads(path.read_text())
        if summary.get('synthetic'):
            raise ValueError('Synthetic smoke results cannot be used in a reproduction report.')
        if summary['experiment'] == 'lr_range':
            continue
        meta = json.loads((path.parent / 'metadata.json').read_text())
        runs.append((path.parent, summary, meta))
    figures = root / 'figures'
    figures.mkdir(parents=True, exist_ok=True)

    for path in root.glob('lr_range_seed*/lr_range.csv'):
        with path.open() as f:
            records = list(csv.DictReader(f))
        fig, axes = plt.subplots(1, 2, figsize=(11, 4))
        axes[0].plot([float(r['lr']) for r in records], [float(r['smooth_loss']) for r in records])
        axes[0].set(xlabel='Learning rate', ylabel='Smoothed training loss', title='LR range test')
        val = [r for r in records if r['val_accuracy']]
        axes[1].plot([float(r['lr']) for r in val], [100*float(r['val_accuracy']) for r in val], 'o-')
        axes[1].set(xlabel='Learning rate', ylabel='Validation accuracy (%)')
        fig.tight_layout()
        fig.savefig(figures / f'{path.parent.name}.png', dpi=180)
        plt.close(fig)
    if not runs:
        print('No completed training runs. LR range plots, if available, were exported.')
        return

    first = runs[0][2]
    for _, _, meta in runs:
        for key in ('source_sha256', 'gpu', 'torch', 'precision', 'split_seed', 'train_size'):
            if meta[key] != first[key]:
                raise ValueError(f'Mixed protocols/devices: {key}. Analyze separate output folders.')
        for key in ('protocol', 'depth', 'batch_size', 'bn_momentum', 'target_val_accuracy'):
            if meta['config'][key] != first['config'][key]:
                raise ValueError(f'Mixed configurations: {key}. Analyze separately.')
    groups = {}
    for _, summary, meta in runs:
        name = summary['experiment']
        if name in groups and groups[name][0][1]['config'] != meta['config']:
            raise ValueError(f'Different settings for {name}. Analyze separately.')
        groups.setdefault(name, []).append((summary, meta))
    output_rows = []
    for name, members in groups.items():
        scores = [100 * s['test_accuracy'] for s, _ in members]
        output_rows.append(dict(experiment=name, seeds=','.join(str(s['seed']) for s, _ in members),
            n=len(scores), test_accuracy_mean_pct=statistics.mean(scores),
            test_accuracy_sd_pp=statistics.stdev(scores) if len(scores) > 1 else '',
            elapsed_seconds_mean=statistics.mean(s['elapsed_seconds'] for s, _ in members),
            updates=members[0][0]['updates'],
            selected_epoch_mean=statistics.mean(s['selected_epoch'] for s, _ in members)))
    with (root / 'aggregate.csv').open('w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=list(output_rows[0]))
        writer.writeheader()
        writer.writerows(output_rows)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
    for path, summary, _ in runs:
        with (path / 'history.csv').open() as f:
            rows = list(csv.DictReader(f))
        label = f'{summary["experiment"]}, seed {summary["seed"]}'
        scores = [100 * float(r['val_accuracy']) for r in rows]
        axes[0].plot([int(r['step']) for r in rows], scores, label=label)
        axes[1].plot([float(r['elapsed_seconds']) / 60 for r in rows], scores, label=label)
    for ax in axes:
        ax.axhline(100 * first['config']['target_val_accuracy'], color='gray', linestyle='--', linewidth=1)
        ax.set(ylabel='Validation accuracy (%)')
        ax.grid(alpha=0.2)
    axes[0].set_xlabel('Optimizer updates')
    axes[1].set_xlabel('Training + validation time (minutes)')
    axes[1].legend(fontsize=7)
    fig.tight_layout()
    fig.savefig(figures / 'convergence.png', dpi=180)
    plt.close(fig)
    paired = []
    indexed = {(s['experiment'], s['seed']): s for _, s, _ in runs}
    for seed in sorted({s['seed'] for _, s, _ in runs}):
        baseline, cycle = indexed.get(('baseline_long', seed)), indexed.get(('onecycle', seed))
        if baseline is None or cycle is None:
            continue
        both_hit = baseline['seconds_to_target'] is not None and cycle['seconds_to_target'] is not None
        paired.append(dict(seed=seed,
            test_delta_pp=100*(cycle['test_accuracy']-baseline['test_accuracy']),
            budget_ratio=baseline['updates']/cycle['updates'],
            time_to_target_ratio=baseline['seconds_to_target']/cycle['seconds_to_target'] if both_hit else None,
            baseline_hit_target=baseline['seconds_to_target'] is not None,
            onecycle_hit_target=cycle['seconds_to_target'] is not None))
    (root / 'paired_comparison.json').write_text(json.dumps(paired, indent=2))
    lines = ['# Measured results', '',
             '| Method | Seeds | Test accuracy, % | SD, pp | Updates | Mean time, min |',
             '|---|---|---:|---:|---:|---:|']
    for r in output_rows:
        sd = f'{r["test_accuracy_sd_pp"]:.2f}' if r['n'] > 1 else 'unavailable (n=1)'
        lines.append(f'| {r["experiment"]} | {r["seeds"]} | {r["test_accuracy_mean_pct"]:.2f} | {sd} | {r["updates"]} | {r["elapsed_seconds_mean"]/60:.2f} |')
    lines.extend(['',
                  'Paired comparisons:', '```json', json.dumps(paired, indent=2), '```', '',
                  'Interpretation must consider test accuracy differences, missing runs and seed variability.'])
    (root / 'results.md').write_text('\n'.join(lines))
    print(f'Exported {root / "aggregate.csv"}, results.md, paired_comparison.json and figures.')


if __name__ == '__main__':
    main()


## Запуск отдельных экспериментов


In [ ]:
os.chdir(WORKDIR)
import torchvision, numpy, matplotlib
print('torchvision:', torchvision.__version__, 'numpy:', numpy.__version__, 'matplotlib:', matplotlib.__version__)

def run_experiment(name, seed, stop_after=None):
    config = json.loads((WORKDIR / 'configs/kaggle.json').read_text())
    expected = {k:v for k,v in config.items() if k not in ('experiments', 'lr_range')}
    expected.update(config['lr_range'] if name == 'lr_range' else config['experiments'][name])
    output = WORKDIR / 'results' / f'{name}_seed{seed}'
    if (output / 'metadata.json').exists():
        previous = json.loads((output / 'metadata.json').read_text())
        if previous['config'] != expected:
            raise RuntimeError(f'Настройки {name} изменились. Выберите новую WORKDIR.')
    if (output / 'summary.json').exists():
        print('Готовый прогон:', name, seed)
        return
    command = [sys.executable, '-m', 'src.train', '--experiment', name, '--seed', str(seed),
               '--data-root', str(DATA_ROOT), '--device', 'cuda:0']
    if (output / 'last.pt').exists():
        command += ['--resume']
    elif output.exists() and any(output.iterdir()):
        raise RuntimeError(f'Незавершённый запуск без checkpoint: {output}. Сохраните логи и используйте другую папку.')
    if stop_after is not None:
        command += ['--stop-after-epoch', str(stop_after)]
    subprocess.run(command, check=True)


## 1. LR range test
Линейный рост LR от 0.01 до 2.0 за 1000 шагов. Validation измеряется каждые 50 шагов. При расходимости тест остановится. Его веса не переходят в основные эксперименты.

Источник идеи: статья, раздел 3. Границы 0.01–2.0, 1000 шагов и проверка validation каждые 50 шагов выбраны для нашего запуска; это не значения из статьи.


In [ ]:
run_experiment('lr_range', 42)
subprocess.run([sys.executable, '-m', 'src.analyze'], check=True)
from IPython.display import display, Image, FileLink, Markdown
display(Image(filename=str(WORKDIR / 'results/figures/lr_range_seed42.png')))


## 2. Выбор LR по отдельному пилоту
Правило задано до основного обучения: найти лучшую validation accuracy среди измерений с LR ≥ 0.2; при равенстве выбрать меньший LR. Взять 80% этого LR и ограничить максимум диапазоном [0.2, 1.0]. Это эвристика, а не гарантия super-convergence. Выбор сохраняется в `lr_selection.json` и используется для всех трёх seeds без повторного подбора.

Правило 80% и ограничение [0.2, 1.0] отсутствуют в статье. В разделе 3 авторы описывают выбор максимума по пику test accuracy в LR range test; наш код использует отдельный validation и другое правило.


In [ ]:
import csv, math




range_path = WORKDIR / 'results/lr_range_seed42/lr_range.csv'
with range_path.open() as file:
    pilot_rows = list(csv.DictReader(file))
candidates = [r for r in pilot_rows if r['val_accuracy'] and float(r['lr']) >= 0.2
              and math.isfinite(float(r['val_accuracy']))]
if not candidates:
    raise RuntimeError('No usable validation measurements in the LR pilot. Inspect its logs.')
pilot_best = max(candidates, key=lambda r: (float(r['val_accuracy']), -float(r['lr'])))
CHOSEN_MAX_LR = round(min(1.0, max(0.2, 0.8 * float(pilot_best['lr']))), 4)
LR_SELECTION_NOTE = (
    'Fixed before the main runs: best validation point of the separate LR range test, '
    f"lr={pilot_best['lr']}, val_accuracy={pilot_best['val_accuracy']}. "
    'Chosen maximum is 80% of this LR, bounded to [0.2, 1.0]. '
    'This is a conservative heuristic, not a guarantee of super-convergence.'
)
config_path = WORKDIR / 'configs/kaggle.json'
config = json.loads(config_path.read_text())
for name in ['onecycle', 'onecycle_high_wd']:
    config['experiments'][name]['lr_max'] = CHOSEN_MAX_LR
    config['experiments'][name]['lr_start'] = CHOSEN_MAX_LR / 10
    config['experiments'][name]['lr_final'] = CHOSEN_MAX_LR / 10000
for name in EXPERIMENTS:
    for seed in SEEDS:
        path = WORKDIR / 'results' / f'{name}_seed{seed}' / 'metadata.json'
        if path.exists():
            prior = json.loads(path.read_text())
            if prior['config']['schedule'] == 'onecycle' and prior['config']['lr_max'] != config['experiments'][name]['lr_max']:
                raise ValueError('LR differs from a started experiment. Use a fresh WORKDIR.')
config_path.write_text(json.dumps(config, indent=2))
(WORKDIR / 'lr_selection.json').write_text(json.dumps(
    {'max_lr':CHOSEN_MAX_LR, 'reason':LR_SELECTION_NOTE, 'pilot_best':pilot_best}, indent=2))
print('Selected maximum LR:', CHOSEN_MAX_LR)
print(LR_SELECTION_NOTE)


## 3. Основное сравнение



Эти эпохи, разделение данных и выбор checkpoint относятся к нашей адаптации. Циклический momentum 0.95–0.85 встречается в статье для wide ResNet и CIFAR-100 (таблица 2), но не является настройкой основного CIFAR-10 / ResNet-56 примера (таблица 8: momentum 0.9).


In [ ]:
for seed in SEEDS:
    for experiment in EXPERIMENTS:
        run_experiment(experiment, seed, STOP_AFTER_EPOCH)


## 4. Метрики и интерпретация
Сравниваем test accuracy и SD по seeds, парную разницу точности, время/шаги до 85% validation accuracy. Порог 85% выбран для нашего анализа и не является критерием статьи. Статистическая эквивалентность методов не установлена.

Отношение бюджетов 120/24 = 5 само по себе не доказывает ускорение. Если порог не достигнут, время до него равно null. Графики относятся к validation. Train accuracy измеряется на аугментированных батчах во время обновления весов, поэтому её разница с validation не является точной оценкой generalization gap.


In [ ]:
subprocess.run([sys.executable, '-m', 'src.analyze'], check=True)
report = WORKDIR / 'results/results.md'
if report.exists():
    display(Markdown(report.read_text()))
chart = WORKDIR / 'results/figures/convergence.png'
if chart.exists():
    display(Image(filename=str(chart)))


## 5. Сохранение


In [ ]:
archive_base = Path('/kaggle/working/hw2_kaggle_results')
archive_path = shutil.make_archive(str(archive_base), 'zip', root_dir=WORKDIR.parent, base_dir=WORKDIR.name)
display(FileLink(archive_path))
print('Сохранено:', archive_path)
